In [ ]:
!pip install transformers datasets scikit-learn torch


In [ ]:
import torch
import json
import numpy as np
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForSequenceClassification, Trainer, TrainingArguments
from sklearn.metrics import accuracy_score, precision_recall_fscore_support

def compute_metrics(pred):
    labels = pred.label_ids
    preds = pred.predictions.argmax(-1)
    precision, recall, f1, _ = precision_recall_fscore_support(labels, preds, average='weighted', zero_division=0)
    acc = accuracy_score(labels, preds)
    return {'accuracy': acc, 'f1': f1, 'precision': precision, 'recall': recall}

print('Loading dataset...')
dataset = load_dataset('banking77')
model_name = 'distilbert-base-uncased'
tokenizer = AutoTokenizer.from_pretrained(model_name)

def tokenize_function(examples):
    return tokenizer(examples['text'], padding='max_length', truncation=True)

tokenized_datasets = dataset.map(tokenize_function, batched=True)
num_labels = 77
model = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=num_labels)

tokenized_datasets = tokenized_datasets.remove_columns(['text'])
tokenized_datasets = tokenized_datasets.rename_column('label', 'labels')
tokenized_datasets.set_format('torch')

training_args = TrainingArguments(
    output_dir='./results',
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=3,
    weight_decay=0.01,
    evaluation_strategy='epoch',
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_datasets['train'],
    eval_dataset=tokenized_datasets['test'],
    compute_metrics=compute_metrics,
)

print('Training...')
trainer.train()

print('Evaluating...')
eval_results = trainer.evaluate()
with open('metrics.json', 'w') as f:
    json.dump(eval_results, f)

print('Saving weights to .pth and labels...')
torch.save(model.state_dict(), 'banking77_distilbert.pth')
label_names = dataset['train'].features['label'].names
with open('labels.json', 'w') as f:
    json.dump(label_names, f)
print('Done! Download banking77_distilbert.pth, metrics.json, and labels.json')
